In [1]:
import pandas as pd
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt 
from tensorflow import keras
from tensorflow.keras import layers, callbacks

2025-02-09 21:19:18.103649: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
df = pd.read_csv('training_dataset.csv')

In [23]:
# merchant_category = {"Groceries" : 0,  'Bank' : 1, 'Electronics' : 2,  'ATM' : 3, 'Restaurant' : 4, 'Luxury Goods' : 5}
#     # status = {'Success' : 0, 'Pending' : 1}
# df['merchant_category'] = df['merchant_category'].map(merchant_category)
# df[['merchant_category']].head()

In [3]:
import re
# Function to extract latitude and longitude
def extract_lat_lon(coord):
    matches = re.findall(r"[-+]?\d*\.\d+|\d+", str(coord))  # Extract all numeric values
    if len(matches) == 2:
        return float(matches[0]), float(matches[1])  # Convert to float
    return None, None  # Return None if values are missing

In [4]:
from sklearn.preprocessing import MinMaxScaler

def preprocessing(df):
    
    drop_columns = [
    'transaction_id', 'user_id', 'merchant_id', 'device_id', 
    'ip_address', 'browser_fingerprint', 'currency','risk_score', 'status', 'is_new_device'
    ]
    df_clean = df.drop(columns=drop_columns, axis=1)
    df_clean['timestamp'] = pd.to_datetime(df_clean['timestamp'])
    df_clean['hour'] = df_clean['timestamp'].dt.hour
    df_clean['day_of_week'] = df_clean['timestamp'].dt.dayofweek
    df_clean['is_weekend'] = df_clean['timestamp'].dt.weekday // 5
    df_clean[['lat', 'lon']] = df_clean['geolocation'].apply(lambda x: pd.Series(extract_lat_lon(x)))
    # df_clean['is_new_device'] = df_clean['is_new_device'].astype(int)
    merchant_category = {"Groceries" : 0,  'Bank' : 1, 'Electronics' : 2,  'ATM' : 3, 'Restaurant' : 4, 'Luxury Goods' : 5}
    status = {'Success' : 0, 'Pending' : 1}


    df_clean['merchant_category'] = df_clean['merchant_category'].map(merchant_category)
    # status = {'Success' : 0, 'Pending' : 1}
    # df_clean['status'] = df_clean['status'].map(status)
    transaction_type = {'Refund' : 0, 'Transfer' : 1, 'Purchase' : 2, 'Withdrawal' : 3}

    df_clean['transaction_type'] = df_clean['transaction_type'].map(transaction_type)
    df_clean = df_clean.drop(['timestamp', 'geolocation'], axis=1)
    df_clean['transaction_ratio'] = df_clean['amount'] / df_clean['balance_before']

    scaler = MinMaxScaler()
    df_clean['amount'] = np.log1p(df_clean['amount'])
    columns_to_normalize = ['balance_before', 'amount', 'transaction_velocity', 'geo_velocity', 'lat', 'lon', 'transaction_ratio']
    df_clean[columns_to_normalize] = scaler.fit_transform(df_clean[columns_to_normalize])
    
    return df_clean

In [5]:
df_clean = preprocessing(df)

In [6]:
import pandas as pd

# Assuming your DataFrame is named df
correlation_matrix = df_clean.corr()

# Extract correlation with 'is_fraudulent'
correlation_with_target = correlation_matrix['is_fraudulent'].abs().sort_values(ascending=False)

print(correlation_with_target)

is_fraudulent           1.000000
lon                     0.985010
merchant_category       0.842679
amount                  0.610847
transaction_ratio       0.577774
hour                    0.310750
transaction_type        0.301717
transaction_velocity    0.167997
lat                     0.028193
balance_before          0.001566
geo_velocity            0.000614
is_weekend              0.000523
day_of_week             0.000088
Name: is_fraudulent, dtype: float64


In [7]:
# Assuming you want to drop columns with correlation < 0.1
columns_to_drop = correlation_with_target[correlation_with_target < 0.1].index

# Drop these columns from the DataFrame
df_clean = df_clean.drop(columns=columns_to_drop)

print("Columns dropped:", columns_to_drop)
print("Remaining columns:", df_clean.columns)

Columns dropped: Index(['lat', 'balance_before', 'geo_velocity', 'is_weekend', 'day_of_week'], dtype='object')
Remaining columns: Index(['transaction_type', 'amount', 'merchant_category', 'is_fraudulent',
       'transaction_velocity', 'hour', 'lon', 'transaction_ratio'],
      dtype='object')


In [10]:
# from sklearn.model_selection import train_test_split

# def split_data(df_clean):

#     Y = df_clean['is_fraudulent']
#     X = df_clean.drop(columns=['is_fraudulent'], axis=1)   
#     X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.1, random_state=42, stratify=Y)

#     # Print dataset sizes
#     print(f"Training Set: {X_train.shape[0]} rows")
#     print(f"Testing Set: {X_test.shape[0]} rows")  

#     return X_train, X_test, y_train, y_test 
  
# X_train, X_test, y_train, y_test = split_data(df_clean)
# print(f"X_train : {X_train.shape}")
# print(f"X_train : {X_test.shape}")


In [11]:

Y = df_clean['is_fraudulent']
X = df_clean.drop(columns=['is_fraudulent'], axis=1)

In [12]:
from tensorflow.keras.optimizers import Adam

# Define the ANN model
def build_ann(learning_rate=0.001, neurons=128, activation='relu'):
    model = keras.Sequential([
        layers.Dense(neurons, activation=activation, input_shape=(X_train.shape[1],)),  # Input layer
        layers.BatchNormalization(),  # Normalize activations
        layers.Dropout(0.3),  # Reduce overfitting

        layers.Dense(neurons, activation=activation),
        layers.BatchNormalization(),
        layers.Dropout(0.3),

        layers.Dense(neurons, activation=activation),
        layers.BatchNormalization(),
        layers.Dropout(0.2),

        layers.Dense(1, activation='sigmoid')  # Output layer (Binary Classification)
    ])
    
    # Compile model
    model.compile(optimizer=Adam(learning_rate), loss='binary_crossentropy', metrics=['accuracy'])
    return model

# Instantiate the model
# model = build_ann(learning_rate=0.001, neurons=128, activation='relu')

In [18]:
from sklearn.model_selection import KFold

# Define K-Fold Cross-Validation
k = 5
kf = KFold(n_splits=k, shuffle=True, random_state=42)
# Store scores for each fold
fold_accuracies = []

# Perform K-Fold Cross-Validation
for train_index, test_index in kf.split(X):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = Y.iloc[train_index], Y.iloc[test_index]

    # Create a new ANN model for each fold
    model = build_ann()

    # Train model
    model.fit(X_train, y_train, epochs=10, batch_size=64, verbose=0)

    # Evaluate model
    _, acc = model.evaluate(X_test, y_test, verbose=0)
    fold_accuracies.append(acc)
    print(f"Fold Accuracy: {acc:.4f}")

/Users/UZAIR/Desktop/fraud_detection/venv/lib/python3.12/site-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Fold Accuracy: 1.0000


/Users/UZAIR/Desktop/fraud_detection/venv/lib/python3.12/site-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Fold Accuracy: 1.0000


/Users/UZAIR/Desktop/fraud_detection/venv/lib/python3.12/site-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Fold Accuracy: 1.0000


/Users/UZAIR/Desktop/fraud_detection/venv/lib/python3.12/site-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Fold Accuracy: 1.0000


/Users/UZAIR/Desktop/fraud_detection/venv/lib/python3.12/site-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Fold Accuracy: 1.0000


In [14]:
X.shape

(500000, 7)

In [16]:
X.columns

Index(['transaction_type', 'amount', 'merchant_category',
       'transaction_velocity', 'hour', 'lon', 'transaction_ratio'],
      dtype='object')

In [ ]:
# Early stopping to prevent overfitting
early_stop = callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

# Train the model
history = model.fit(X_train, y_train, epochs=10, batch_size=64, validation_split=0.2, callbacks=[early_stop], verbose=1)

In [20]:
# import keras_tuner as kt

# tuner = kt.Hyperband(
#     build_ann,
#     objective='val_accuracy',
#     max_epochs=50,
#     factor=3,  # How aggressive to be with pruning
#     directory='tuner_results',
#     project_name='ann_tuning'
# )

In [21]:
# tuner.search(X_train, y_train, epochs=50, validation_split=0.2, verbose=1)


In [22]:
# best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]
# print(f"Best Hyperparameters: {best_hps.values}")


In [23]:
import keras_tuner as kt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import make_classification

# Generate a sample dataset
X, y = make_classification(n_samples=5000, n_features=20, random_state=42)

# Normalize data
scaler = StandardScaler()
X = scaler.fit_transform(X)

# Split dataset into training and testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Define the ANN model with HyperParameters
def build_ann_model(hp):
    model = keras.Sequential([
        layers.Dense(hp.Int('neurons', min_value=32, max_value=256, step=32), 
                     activation=hp.Choice('activation', ['relu', 'tanh']), 
                     input_shape=(X_train.shape[1],)),
        layers.BatchNormalization(),
        layers.Dropout(hp.Float('dropout_rate', min_value=0.2, max_value=0.5, step=0.1)),

        layers.Dense(hp.Int('neurons', min_value=32, max_value=256, step=32), 
                     activation=hp.Choice('activation', ['relu', 'tanh'])),
        layers.BatchNormalization(),
        layers.Dropout(hp.Float('dropout_rate', min_value=0.2, max_value=0.5, step=0.1)),

        layers.Dense(1, activation='sigmoid')
    ])
    
    # Compile model with a tunable learning rate
    model.compile(optimizer=Adam(learning_rate=hp.Choice('learning_rate', [0.001, 0.0005, 0.0001])), 
                  loss='binary_crossentropy', 
                  metrics=['accuracy'])
    
    return model

# Define the tuner using Hyperband
tuner = kt.Hyperband(
    build_ann_model,
    objective='val_accuracy',
    max_epochs=50,
    factor=3,  
    directory='tuner_results',
    project_name='ann_tuning'
)

# Run hyperparameter search
tuner.search(X_train, y_train, epochs=50, validation_split=0.2, verbose=1)

# Get best model
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]
print(f"Best Hyperparameters: {best_hps.values}")

# Train the best model
best_model = tuner.hypermodel.build(best_hps)
best_model.fit(X_train, y_train, epochs=100, validation_split=0.2, verbose=1)

# Evaluate on test data
test_loss, test_acc = best_model.evaluate(X_test, y_test)
print(f"Test Accuracy: {test_acc:.4f}")


Trial 90 Complete [00h 00m 17s]
val_accuracy: 0.8837500214576721

Best val_accuracy So Far: 0.8887500166893005
Total elapsed time: 00h 08m 58s
Best Hyperparameters: {'neurons': 256, 'activation': 'tanh', 'dropout_rate': 0.30000000000000004, 'learning_rate': 0.0001, 'tuner/epochs': 17, 'tuner/initial_epoch': 6, 'tuner/bracket': 3, 'tuner/round': 2, 'tuner/trial_id': '0034'}
Epoch 1/100
100/100 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.6861 - loss: 0.6227 - val_accuracy: 0.8775 - val_loss: 0.3393
Epoch 2/100
100/100 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8112 - loss: 0.4289 - val_accuracy: 0.8687 - val_loss: 0.3127
Epoch 3/100
100/100 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8147 - loss: 0.4267 - val_accuracy: 0.8775 - val_loss: 0.3185
Epoch 4/100
100/100 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8335 - loss: 0.3869 - val_accuracy: 0.8800 - val_loss: 0.3175
Epoch 5/100
100/100 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8213 - loss: 0.4200 - val_accuracy: 0.877